# Seleção de Atributos com Chi-Quadrado (SelectKBest)

Neste notebook, vamos aprender como a **Seleção de Atributos** (ou *Feature Selection*) pode melhorar o desempenho e a velocidade de modelos de Machine Learning. Veremos na prática como reduzir a dimensionalidade de uma base de dados complexa usando testes estatísticos, mantendo apenas as informações mais relevantes para a nossa previsão.

## Objetivos de Aprendizagem
* Entender o conceito de **Seleção de Atributos** e por que menos variáveis podem gerar um modelo melhor.
* Compreender o funcionamento do teste estatístico **Chi-Quadrado ($\chi^2$)** aplicado à filtragem de dados.
* Comparar o desempenho (acurácia) de um modelo Bayesiano treinado com a base completa versus a base filtrada.
* Analisar quais atributos foram selecionados como os mais importantes.

## Pré-requisitos e Bibliotecas Utilizadas
* **Pandas (`pd`)**: Usado para carregar e manipular nossos dados em tabelas (DataFrames).
* **Scikit-Learn (`sklearn`)**:
  * `train_test_split`: Para dividir nossos dados em treino e teste, garantindo uma avaliação justa do modelo.
  * `GaussianNB` (Naive Bayes): Nosso algoritmo de classificação para prever se um elemento é um anúncio ou não.
  * `accuracy_score`: Métrica para avaliar a porcentagem de acertos do nosso modelo.
  * `chi2`: O teste estatístico de Chi-Quadrado, usado para avaliar a dependência entre as variáveis e a classe.
  * `SelectKBest`: O seletor que filtra as $K$ melhores variáveis com base no teste estatístico escolhido.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import chi2, SelectKBest

### Passo 1: Importação dos Dados

**O que este passo faz:** Carrega o arquivo `ad.data` (que contém dados sobre imagens de páginas web, classificando-as em anúncios ou não-anúncios) em um DataFrame do Pandas.

**Por que ele é necessário:** Antes de aplicar qualquer algoritmo, precisamos trazer nossos dados para a memória. Como o arquivo original não possui uma linha de cabeçalho com os nomes das colunas, passamos o parâmetro `header=None` para evitar que a primeira linha de dados seja lida incorretamente como nome de coluna.

**Como interpretar a saída:** A saída `(3279, 1559)` nos indica o formato da tabela: temos **3279 linhas** (amostras) e **1559 colunas** (atributos). É uma quantidade enorme de variáveis para poucas linhas, o que torna este cenário perfeito para a seleção de atributos!

In [ ]:
anuncio = pd.read_csv('ad.data', header=None)
anuncio.shape

(3279, 1559)

### Passo 2: Separação de Variáveis Independentes (X) e Dependentes (y)

**O que este passo faz:** Divide nosso DataFrame em dois arrays: `X`, contendo todos os atributos previsores (todas as colunas menos a última), e `y`, contendo apenas a classe alvo (a última coluna).

**Por que ele é necessário:** Algoritmos de aprendizado supervisionado precisam receber as características de entrada (`X`) separadas da resposta correta que devem aprender a prever (`y`).

💡 **Para pensar:** O fatiamento foi feito usando `.values` diretamente. Se houver valores ausentes (como `?` comuns nessa base de dados) representados como strings, eles podem ser tratados como dados categóricos ou gerar problemas no treinamento. O ideal seria verificar o tipo dos dados antes de convertê-los em arrays puros.

In [ ]:
X = anuncio.iloc[:,:-1].values
y = anuncio.iloc[:,-1].values

In [ ]:
X

array([[125.    , 125.    ,   1.    , ...,   0.    ,   0.    ,   0.    ],
       [ 57.    , 468.    ,   8.2105, ...,   0.    ,   0.    ,   0.    ],
       [ 33.    , 230.    ,   6.9696, ...,   0.    ,   0.    ,   0.    ],
       ...,
       [ 23.    , 120.    ,   5.2173, ...,   0.    ,   0.    ,   0.    ],
       [  0.    ,   0.    ,   0.    , ...,   0.    ,   0.    ,   0.    ],
       [ 40.    ,  40.    ,   1.    , ...,   0.    ,   0.    ,   0.    ]])

In [ ]:
y

array(['ad.', 'ad.', 'ad.', ..., 'nonad.', 'nonad.', 'nonad.'],
      dtype=object)

### Passo 3: Divisão em Treino e Teste (Base Completa)

**O que este passo faz:** Separa os dados originais em duas porções: 70% para treinar o algoritmo (`X_treinamento` e `y_treinamento`) e 30% para testar sua capacidade de generalização (`X_teste` e `y_teste`). O `random_state=0` garante que a divisão seja sempre a mesma ao reexecutar o código.

**Por que ele é necessário:** Precisamos avaliar o modelo em dados que ele nunca viu durante o treinamento para saber se ele realmente aprendeu ou apenas decorou os exemplos (fenômeno chamado de *Overfitting*).

In [ ]:
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(X,y, test_size = 0.3, random_state=0)

### Passo 4: Treinamento e Avaliação do Primeiro Modelo

**O que este passo faz:** Cria uma instância do classificador **Naive Bayes Gaussiano** (`GaussianNB`), treina-o com os dados de treinamento completos e calcula a acurácia das previsões na base de teste.

**Por que ele é necessário:** Estabelece a nossa **linha de base** (baseline). Precisamos saber quão bem o modelo performa usando todos os 1558 atributos de entrada para termos um comparativo.

**Como interpretar a saída:** A acurácia obtida foi de aproximadamente **77.6%**. Embora pareça aceitável, o modelo está lidando com ruído de muitas variáveis irrelevantes.

In [ ]:
modelo1 = GaussianNB()
modelo1.fit(X_treinamento,y_treinamento)
previsoes1 = modelo1.predict(X_teste)
print(accuracy_score(y_teste,previsoes1))

0.7764227642276422


### Passo 5: Seleção de Atributos com SelectKBest e Chi-Quadrado

**O que este passo faz:** Configura o seletor para extrair apenas as $K=7$ variáveis mais importantes usando o teste do **Chi-Quadrado ($\chi^2$)** e aplica a transformação na base original `X` gerando `X_novo`.

**Definição Técnica:** O **Chi-Quadrado** mede a dependência entre variáveis categóricas. Na prática, ele analisa se a presença ou ausência de uma característica está estatisticamente relacionada ao fato de a imagem ser ou não um anúncio. O `SelectKBest` ordena essas pontuações e filtra as melhores.

**Por que ele é necessário:** Reduz radicalmente a complexidade do problema de 1558 variáveis para apenas 7, eliminando ruídos, diminuindo o consumo de memória e o tempo de processamento.

In [ ]:
selecao = SelectKBest(chi2,k=7)
X_novo = selecao.fit_transform(X,y)

In [ ]:
X_novo

array([[125.    , 125.    ,   1.    , ...,   1.    ,   0.    ,   1.    ],
       [ 57.    , 468.    ,   8.2105, ...,   1.    ,   0.    ,   0.    ],
       [ 33.    , 230.    ,   6.9696, ...,   0.    ,   0.    ,   0.    ],
       ...,
       [ 23.    , 120.    ,   5.2173, ...,   0.    ,   0.    ,   0.    ],
       [  0.    ,   0.    ,   0.    , ...,   0.    ,   0.    ,   0.    ],
       [ 40.    ,  40.    ,   1.    , ...,   0.    ,   0.    ,   0.    ]])

In [ ]:
X_novo.shape

(3279, 7)

In [ ]:
print(selecao.get_support())

[ True  True  True ... False False False]


### Passo 6: Divisão em Treino/Teste e Treinamento do Segundo Modelo

**O que este passo faz:** Divide a base simplificada (`X_novo`) em treino e teste usando as mesmas proporções de antes e treina um novo modelo Naive Bayes Gaussiano.

**Por que ele é necessário:** Permite validar se o modelo performa melhor, pior ou igual após a remoção de mais de 1500 colunas.

**Como interpretar a saída:** A nova acurácia disparou para **94.5%**!

**Por que isso aconteceu?** O Naive Bayes assume que todos os atributos são independentes entre si (daí o nome "Naive" / ingênuo). Na base original, havia muitas variáveis correlacionadas ou irrelevantes que confundiam a probabilidade do algoritmo. Ao filtrar apenas as 7 melhores, removemos o "barulho", permitindo que o classificador focasse no que realmente importa.

In [ ]:
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(X_novo,y, test_size = 0.3, random_state=0)

In [ ]:
modelo2 = GaussianNB()
modelo2.fit(X_treinamento,y_treinamento)
previsoes2 = modelo2.predict(X_teste)
print(accuracy_score(y_teste,previsoes2))

0.9451219512195121


## Erros Comuns e Dicas

* **Valores Ausentes ocunlto:** A base de dados `ad.data` é conhecida por ter caracteres como `?` representando valores ausentes em algumas colunas numéricas. Tentar rodar operações matemáticas ou seletores sem tratar esses caracteres pode gerar erros de tipo (*ValueError*).
* **Divisão de treino/teste antes da seleção:** 💡 **Para pensar:** No código, a seleção de atributos (`SelectKBest.fit_transform`) foi feita na base inteira `X` antes de separarmos em treino e teste para o segundo modelo. Isso causa o que chamamos de **Data Leakage** (Vazamento de Dados), pois as estatísticas do conjunto de teste influenciaram a escolha dos melhores atributos de treino. O correto é ajustar o seletor apenas no conjunto de treino e aplicá-lo no de teste!

## Resumo do Aprendizado
* **Menos é Mais:** Reduzir o número de atributos de 1558 para 7 aumentou a acurácia de 77.6% para 94.5%.
* **Filtros Estatísticos:** O método Chi-Quadrado avalia rapidamente a relevância de cada coluna individualmente em relação ao nosso objetivo.
* **Desempenho:** Modelos mais simples treinam muito mais rápido, gastam menos memória e são mais fáceis de interpretar.

## Exercícios Propostos
1. **Fácil:** Altere o valor de $K$ no `SelectKBest` para $15$ e veja se a acurácia do modelo melhora ou piora em relação a 7.
2. **Médio:** Implemente o tratamento de dados ausentes para substituir caracteres `?` por valores válidos (como a média ou mediana da coluna) antes de rodar o algoritmo.
3. **Desafiador:** Altere a ordem de execução aplicando o `SelectKBest` exclusivamente no conjunto de treino (`X_treinamento`) e depois transforme o conjunto de teste (`X_teste`), corrigindo o vazamento de dados apontado na seção "Para pensar".

## Para ir além
* Pesquise sobre métodos de seleção de atributos do tipo **Wrapper** (como *Recursive Feature Elimination - RFE*).
* Entenda a diferença entre métodos de **Filtro** (como Chi-Quadrado) e métodos **Embutidos** (como regularização L1/Lasso em Regressão Logística).